# Using commit vith onnx

In [2]:
from utils_dino_final_pylib import *
import onnxruntime
import time

def get_onnx_mask_from_bboxes_onnx(bboxes, ort_session):

    all_masks = []

    for bbox in bboxes:

        onnx_box_coords = bbox.reshape(2, 2)
        onnx_box_labels = np.array([2,3])

        onnx_coord = np.concatenate([ onnx_box_coords], axis=0)[None, :, :]
        onnx_label = np.concatenate([ onnx_box_labels], axis=0)[None, :].astype(np.float32)

        onnx_coord = predictor.transform.apply_coords(onnx_coord, image.shape[:2]).astype(np.float32)
        
        print("onnx_coord.shape", onnx_coord.shape)
        print("onnx_label.shape", onnx_label.shape)

        print(image.shape[:2])

        onnx_mask_input = np.zeros((1, 1, 256, 256), dtype=np.float32)
        onnx_has_mask_input = np.zeros(1, dtype=np.float32)


        ort_inputs = {
            "image_embeddings": image_embedding,
            "point_coords": onnx_coord,
            "point_labels": onnx_label,
            "mask_input": onnx_mask_input,
            "has_mask_input": onnx_has_mask_input,
            "orig_im_size": np.array(image.shape[:2], dtype=np.float32)
        }

        masks, _, _ = ort_session.run(None, ort_inputs)
        masks = masks > predictor.model.mask_threshold
        print("masks.shape", masks.shape)

        all_masks.append(masks)

    return all_masks

c:\Users\hci\.conda\envs\dinopy\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\hci\.conda\envs\dinopy\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
c:\Users\hci\.conda\envs\dinopy\lib\site-packages\torch\functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3610.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Using device: cuda
final text_encoder_type: bert-base-uncased


c:\Users\hci\.conda\envs\dinopy\lib\site-packages\groundingdino\util\inference.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_checkpoint_

In [3]:
import os
import time
import json
import shutil

start_time = time.time()

FOLDER_PATH = r"D:\3d-recon\WESHAPE-HQ"
OUTPUT_BASE = r"D:\3d-recon\WESHAPE-HQ"

# Loop through all images in folder
for IMAGE_NAME in os.listdir(FOLDER_PATH):
    SOURCE_IMAGE_PATH = os.path.join(FOLDER_PATH, IMAGE_NAME)

    # Skip non-files (like subfolders)
    if not os.path.isfile(SOURCE_IMAGE_PATH):
        continue

    # Remove extension for folder naming
    image_base = os.path.splitext(IMAGE_NAME)[0]

    # Output directory
    output_dir = os.path.join(OUTPUT_BASE, image_base)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\nProcessing: {IMAGE_NAME}")

    # Run wall detection
    wall_bboxes, wall_selected_points = get_wall_bboxes_points(SOURCE_IMAGE_PATH)
    print("wall_bboxes", wall_bboxes)
    print("wall_selected_points", wall_selected_points)

    # Run floor+rug detection
    (floor_bboxes,
     floor_selected_points,
     rug_bboxes
     ) = get_floor_bboxes_points_with_rug(SOURCE_IMAGE_PATH)

    print("floor_bboxes", floor_bboxes)
    print("floor_selected_points", floor_selected_points)
    print("rug_bboxes", rug_bboxes)

    # Build output dict
    output = {
        'wall_bboxes': wall_bboxes.tolist(),
        'wall_selected_points': wall_selected_points,
        'floor_bboxes': floor_bboxes.tolist(),
        'floor_selected_points': floor_selected_points,
        # 'rug_bboxes': rug_bboxes,
    }

    # Save input image as input.jpg
    target_image_path = os.path.join(output_dir, "input.jpg")
    shutil.copy(SOURCE_IMAGE_PATH, target_image_path)

    # Save output dict as auto_detect.json
    target_json_path = os.path.join(output_dir, "auto_detect.json")
    with open(target_json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)

    print(f"✅ Saved results to {output_dir}")
    print("DONE GETTING BBOXES", time.time() - start_time)


Processing: 0_Render_BaseColor.png
Getting wall mask from  D:\3d-recon\WESHAPE-HQ\0_Render_BaseColor.png


c:\Users\hci\.conda\envs\dinopy\lib\site-packages\transformers\modeling_utils.py:1614: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
c:\Users\hci\.conda\envs\dinopy\lib\site-packages\transformers\models\bert\modeling_bert.py:407: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
c:\Users\hci\.conda\envs\dinopy\lib\site-packages\torch\_dynamo\eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return 

WALL SCORES [0.6203202  0.5440576  0.29873556]
wall_bboxes [[1.75628662e+00 1.84506653e+02 6.67120422e+02 8.95240540e+02]
 [8.05800415e+02 1.05070435e+02 1.91733606e+03 9.41820312e+02]
 [6.60046570e+02 2.48535995e+02 8.42693542e+02 7.48732788e+02]]
wall_selected_points [[319, 544], [1430, 532], [735, 489]]
RUG:  0
No overlap
RUG BBOX 3.5082397 104.48636 1916.1096 945.07214
FLOOR BBOX 1.4817505 742.8673 1918.606 1079.2584
RUG:  1
No overlap
RUG BBOX 1.8075867 186.10233 667.9939 893.85913
FLOOR BBOX 1.4817505 742.8673 1918.606 1079.2584
RUG:  2
No overlap
RUG BBOX 801.2065 105.93237 1917.907 941.0359
FLOOR BBOX 1.4817505 742.8673 1918.606 1079.2584
floor_bboxes [[   2.4817505  743.8673    1917.606     1078.2584   ]]
floor_selected_points [[838, 1052]]
rug_bboxes []
✅ Saved results to D:\3d-recon\WESHAPE-HQ\0_Render_BaseColor
DONE GETTING BBOXES 3.408799171447754

Processing: 0_Render_Cavity.png
Getting wall mask from  D:\3d-recon\WESHAPE-HQ\0_Render_Cavity.png
WALL SCORES [0.3778647  0.3

In [ ]:
import os
import time
import json
import shutil

start_time = time.time()

# Input directories
ASAROOM_PATH = r"D:\3d-recon\datasets\ASARoomImage"
PRESET_PATH = r"D:\3d-recon\WESHAPE-COTTO-PRESET\PresetPreparation\all_preset_data"

# Output directory
OUTPUT_BASE = r"D:\3d-recon\WESHAPExZIM\input-autodetect-json_preset"

def process_image(image_path, image_base, output_dir):
    """Run wall/floor/rug detection and save results"""
    print(f"\nProcessing: {image_path}")

    # Run wall detection
    wall_bboxes, wall_selected_points = get_wall_bboxes_points(image_path)
    print("wall_bboxes", wall_bboxes)
    print("wall_selected_points", wall_selected_points)

    # Run floor+rug detection
    (floor_bboxes,
     floor_selected_points,
     rug_bboxes
     ) = get_floor_bboxes_points_with_rug(image_path)

    print("floor_bboxes", floor_bboxes)
    print("floor_selected_points", floor_selected_points)
    print("rug_bboxes", rug_bboxes)

    # Build output dict
    output = {
        'wall_bboxes': wall_bboxes.tolist(),
        'wall_selected_points': wall_selected_points,
        'floor_bboxes': floor_bboxes.tolist(),
        'floor_selected_points': floor_selected_points,
        # 'rug_bboxes': rug_bboxes,
        # 'rug_to_floor_indices': rug_to_floor_indices
    }

    # Save input image as input.jpg
    target_image_path = os.path.join(output_dir, "input.jpg")
    shutil.copy(image_path, target_image_path)

    # Save output dict as auto_detect.json
    target_json_path = os.path.join(output_dir, "auto_detect.json")
    with open(target_json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)

    print(f"✅ Saved results to {output_dir}")
    print("DONE GETTING BBOXES", time.time() - start_time)


# ---------------------------
# Pass 1: images in ASARoomImage
# ---------------------------
# for IMAGE_NAME in os.listdir(ASAROOM_PATH):
#     SOURCE_IMAGE_PATH = os.path.join(ASAROOM_PATH, IMAGE_NAME)
#     if not os.path.isfile(SOURCE_IMAGE_PATH):
#         continue

#     image_base = os.path.splitext(IMAGE_NAME)[0]
#     output_dir = os.path.join(OUTPUT_BASE, image_base)
#     os.makedirs(output_dir, exist_ok=True)

#     process_image(SOURCE_IMAGE_PATH, image_base, output_dir)


# ---------------------------
# Pass 2: images in all_preset_data subfolders
# ---------------------------
for folder in os.listdir(PRESET_PATH):
    folder_path = os.path.join(PRESET_PATH, folder)
    if not os.path.isdir(folder_path):
        continue

    # Image should have same name as folder
    image_path = os.path.join(folder_path, f"{folder}.jpg")
    if not os.path.exists(image_path):
        print(f"⚠️ No matching image found in {folder_path}")
        continue

    output_dir = os.path.join(OUTPUT_BASE, folder)
    os.makedirs(output_dir, exist_ok=True)

    process_image(image_path, folder, output_dir)